In [4]:
import sys
!{sys.executable} -m pip install -r requirements.txt

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.1 MB/s eta 0:00:00
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 22.0 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 0.21.0
    Uninstalling python-dotenv-0.21.0:
      Successfully uninstalled python-dotenv-0.21.0


In [ ]:
# importing necessary libraries and loading environment variables
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"

print("Ready!")

Ready!


In [ ]:
# Function to scrape job posting content and count tokens
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

def scrape_job_posting(url):
    try:
        response = requests.get(url, headers= HEADERS, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:5000]
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return ""

def count_tokens(text):
    encoding = tiktoken.encoding_for_model(MODEL)
    return len(encoding.encode(text))


In [7]:
# My Background
My_BACKGROUND = """
Name: Md Qamar Hashmi
Skills: AWS, Kubernetes, Terraform, Python, GitOps, ArgoCD, Helm, CI/CD, GitHub Actions, Jenkins, Prometheus, Grafana, CloudWatch, Incident Response, SRE, Platform Engineering, Observability, Release Engineering, Multi-cloud, Azure, Docker, Linux, Networking, Security, Automation, Infrastructure as Code, DevOps, Cloud Architecture, System Design, Performance Optimization, Cost Management, Disaster Recovery, Compliance, Monitoring, Logging, Alerting, Scalability, High Availability
Experience: 5+ years in DevOps, SRE, and Platform Engineering roles, with a strong focus on cloud infrastructure, automation, and observability. Proven track record of designing and implementing scalable, secure, and cost-effective solutions in multi-cloud environments. Experienced in leading incident response efforts and optimizing system performance.
Education: Bachelor's degree in Information Technology
Certifications: AWS Certified Solutions Associate, Certified Kubernetes Administrator (CKA), Microsoft Certified: Azure Administrator Associate, AWS Certified DevOps Engineer - Professional
"""

system_prompt = """
"You are an expert job analyst. Extract key information and return JSON with: "
"role, company, required_skills (list), nice_to_have (list), "
"experience_years, key_responsibilities (list), culture_keywords (list)."
"""

user_prompt = f"Analyze this job posting:\n\n{job_text}, and compare it to this candidate background:\n\n{My_BACKGROUND} and provide insights on how well the candidate matches the job requirements."

NameError: name 'job_text' is not defined

In [ ]:
# scrape a job posting and count tokens
JOB_URL = "https://www.linkedin.com/jobs/view/..."  # paste a real URL

job_text = scrape_job_posting(JOB_URL)
print(f"Scraped {count_tokens(job_text)} tokens")
print(job_text[:5000])

In [ ]:
# function to analyze job posting and compare with candidate background
def analyze_job(job_text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", 
             "content": (
                 "You are an expert job analyst. Extract key information and return JSON with: "
                "role, company, required_skills (list), nice_to_have (list), "
                "experience_years, key_responsibilities (list), culture_keywords (list)."
             )
            },
            {"role": "user", 
             "content": f"Here is the job posting text:\n{job_text}\n\nPlease analyze it and provide insights on required skills, experience, and how well it matches my background."}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

job_analysis = analyze_job(job_text)
print(json.dumps(job_analysis, indent=2))
    